# IEEE 30 with two embedded HVDC grids

The IEEE 30-bus system with two HVDC grids embedded in it: power can flow between AC buses both over the AC lines and through the DC grids, so the converter setpoints steer the AC flows.

| | |
|---|---|
| AC buses | 30 (41 lines, 0 transformers, 4 generators) |
| DC buses | 8, in 2 DC grids |
| Converters | 6: 3 × VdcQ (holds its DC voltage, fixed Q), 3 × PQ (fixed P and Q) |
| Case file | `validation/cases/ieee30_hvdc.yaml` |

## What this notebook does

1. **Load** the case file into a pandapower net and classify the nodes. `LOSSY = True` adds the converter
   losses a + b·|I| + c·|I|² (coefficients of the paper, section IV-B).
2. **Build Y**: the AC admittance matrix (lines, transformer taps and shifts, shunts) and the DC conductance
   matrix.
3. **Solve** the AC grids, the DC grids and the converters together in one Newton–Raphson, and print the
   iterations, the largest mismatch and the run time.
4. **Results**: per node the voltage E = E_re + j·E_im (p.u.) and the injected power P, Q (p.u., positive into
   the grid); per converter its AC and DC power, losses and current.

## Try it

- Change a converter in the `converter` table of the case file, for example its `mode`, `p_mw`, `q_mvar` or
  `vdc_pu`, then *Run All*. Modes: PQ, VdcQ, PV, VdcV, GF (grid-forming) and droop; the columns each mode uses are
  listed at the top of the case file.
- Set `LOSSY = True` in the first cell to include converter losses.

## Running this notebook

Use the kernel **Python 3.12 (unified-acdc-powerflow)** (VS Code: *Select Kernel* → *Jupyter Kernel…*). To set it
up once, in the repository folder:

```bash
python3.12 -m venv .venv
.venv/bin/pip install -e ".[dev]"
.venv/bin/python -m ipykernel install --user --name unified-acdc-powerflow --display-name "Python 3.12 (unified-acdc-powerflow)"
```


In [1]:
import logging, warnings
logging.getLogger("pandapower").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_rows", 2000)
from unified_acdc_powerflow import build_model, build_ybus, solve
from unified_acdc_powerflow.cases import load_case

# 1) load the case and classify the nodes
CASE = "ieee30_hvdc"
LOSSY = False          # True: converter losses a + b|I| + c|I|^2 (coefficients of the paper)
net = load_case(CASE, lossy=LOSSY)   # pandapower net from validation/cases/ieee30_hvdc.yaml
model = build_model(net)
print(f"{model.n_ac} AC nodes, {model.n_dc} DC nodes, {model.n_conv} converters")
net.vsc[["name", "bus", "bus_dc", "mode", "p_ac_mw", "q_ac_mvar", "vm_dc_pu", "p_dc_ref_mw", "droop_k"]]


30 AC nodes, 8 DC nodes, 6 converters


,name,bus,bus_dc,mode,p_ac_mw,q_ac_mvar,vm_dc_pu,p_dc_ref_mw,droop_k
0,IC 113-1,113,1,VdcQ,0.0,0.2,1.0,0.0,0.0
1,IC 115-2,115,2,VdcQ,0.0,0.2,1.0,0.0,0.0
2,IC 130-3,130,3,PQ,0.1,0.2,1.0,0.0,0.0
3,IC 102-4,102,4,VdcQ,0.0,0.2,1.0,0.0,0.0
4,IC 105-5,105,5,PQ,0.1,0.2,1.0,0.0,0.0
5,IC 106-6,106,6,PQ,0.1,0.2,1.0,0.0,0.0


In [2]:
# 2) build Y (AC: pandapower Ybus with taps and shunts; DC: conductance matrix G)
build_ybus(model)
print("Y (AC):", model.Y.shape, "nnz", model.Y.nnz)
print("G (DC):", model.G.shape, "nnz", model.G.nnz)


Y (AC): (30, 30) nnz 112
G (DC): (8, 8) nnz 22


In [3]:
# 3) Newton-Raphson
res = solve(model)
print(res.message, "| iterations:", res.iterations, "| max mismatch:", f"{res.max_mismatch:.2e}",
      "| time:", f"{res.solve_time_s * 1e3:.1f} ms")


converged | iterations: 4 | max mismatch: 7.30e-15 | time: 57.3 ms


In [4]:
# 4) results: per node E (real, imag) and injected P, Q in p.u.; then per converter
display(res.table())
res.converter_table()


,side,bus,name,type,E_re,E_im,P_pu,Q_pu
0,ac,101,101,slack,1.000000,0.000000,9.512860e-02,3.805033e-02
1,ac,102,102,conv VdcQ,0.999998,0.001927,5.545587e-01,1.287218e-01
2,ac,103,103,PQ,0.983655,-0.020894,-2.400000e-02,-1.200000e-02
3,ac,104,104,PQ,0.980638,-0.024376,-7.600000e-02,-1.600000e-02
4,ac,105,105,conv PQ,0.982543,-0.023736,1.000000e-03,2.000000e-03
5,ac,106,106,conv PQ,0.973338,-0.031463,1.000000e-03,2.000000e-03
6,ac,107,107,PQ,0.967198,-0.037327,-2.280000e-01,-1.090000e-01
7,ac,108,108,PQ,0.960506,-0.038726,-3.000000e-01,-3.000000e-01
8,ac,109,109,PQ,0.980181,-0.043991,1.869663e-16,1.736266e-15
9,ac,110,110,PQ,0.983765,-0.050554,-5.800000e-02,-2.000000e-02


,vsc,name,mode,P_ac_pu,Q_ac_pu,P_dc_pu,loss_pu,I_pu
0,0,IC 113-1,VdcQ,-0.000500,0.002,0.000500,0.0,0.002042
1,1,IC 115-2,VdcQ,-0.000500,0.002,0.000500,0.0,0.002097
2,2,IC 130-3,PQ,0.001000,0.002,-0.001000,0.0,0.002308
3,3,IC 102-4,VdcQ,0.244859,0.002,-0.244859,0.0,0.244867
4,4,IC 105-5,PQ,0.001000,0.002,-0.001000,0.0,0.002275
5,5,IC 106-6,PQ,0.001000,0.002,-0.001000,0.0,0.002296
